In [1]:
# Cell 1 — install pinned Undercurrent + core research packages
%pip install -q "undercurrent==0.1.0" "transformers>=4.45,<5" "accelerate>=1.0,<2" "huggingface_hub>=0.26" "safetensors>=0.4.5" "pandas>=2.2,<3" "numpy>=1.26,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "seaborn>=0.13,<1" "pyarrow>=18,<20"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
bigframes 2.51.0 requires pyarrow>=23.0.1, but you have pyarrow 19.0.1 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
# Cell 2 — mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# IMPORTANT:
# The destination is shared with you. Colab's mounted MyDrive does not always expose
# "Shared with me" as a normal path. In Drive in your browser:
# Shared with me > open the target folder > Organise > Add shortcut > My Drive.
# Add the shortcut inside My Drive/MYSSP (or at My Drive root).
# Then adjust this path after inspecting the printed folders below.
import os
print("Top-level Drive entries:", os.listdir("/content/drive/MyDrive")[:50])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Top-level Drive entries: ['Colab Notebooks', 'MYSSP', 'Project proposal.gdoc', 'Fablehaven.pdf', 'Kayson Public Speaking Draft Review 15Jan2023.docx', 'Copy of Impromptu CheatSheet.gdoc', 'MWSDES Intermediate - 1.1 - Characterisation & Analytical Rigour.pdf', 'MWSDES Intermediate - 1.2 - Layered Rebuttals.pdf', 'MWSDES Intermediate - 1.3 - Advanced Modelling.pdf', 'MWSDES Intermediate - Handout 1.3 - Modelling.gdoc', 'trim.94143CEA-C550-4FFB-A512-35FB8AD0712D.MOV', 'trime15f135c-901f-4d92-bf23-a8d951763e6d_yWZVEWxW.mp4', 'Coding', 'Kayson Goh.pdf', 'Tote Bags Record - Family Day.gsheet', 'Big Arjun motion.gdoc', 'Arjun spar club spar spar.gdoc', 'mnswsdc rounds.gdoc', 'Classroom', 'Student council.gdoc', 'HLB Receipt 330814.pdf', '2c English LANG task.gdoc', "prep for final selections '24 db.gdoc", 'final seelctionmatter test24.gdoc', 'lit.gdoc', 'FEBRUARY TK

In [5]:
# Cell 3 — authenticate without exposing your token
# In Colab: left sidebar > key icon (Secrets) > add secret named HF_TOKEN.
# Enable notebook access for HF_TOKEN. Do NOT paste the token into a code cell.
from google.colab import userdata
from huggingface_hub import login
hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN secret not found. Add it in Colab Secrets and enable notebook access.")

# Strip any accidental leading/trailing/embedded whitespace or return characters
hf_token_cleaned = "".join(hf_token.split())

login(token=hf_token_cleaned, add_to_git_credential=False)
print("Hugging Face login completed; token was not printed.")

Hugging Face login completed; token was not printed.


In [6]:
# Cell 5 — pin and inspect Undercurrent
# First verify the installed version and the CLI. This downloads model config only, not 8B weights.
import importlib.metadata, subprocess, sys
print("Undercurrent version:", importlib.metadata.version("undercurrent"))
assert importlib.metadata.version("undercurrent") == "0.1.0"
MODEL_IDS = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "google/gemma-2-9b-it",
]
for model_id in MODEL_IDS:
    print("\\n" + "=" * 90 + "\\nMODEL:", model_id)
    result = subprocess.run(
        ["undercurrent", "inspect-model", model_id, "--backend", "hf"],
        text=True, capture_output=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("ERROR:", result.stderr)
        print("Record this as a model-access/config issue; do not silently omit the model.")

Undercurrent version: 0.1.0
\n==========================================================================================\nMODEL: meta-llama/Llama-3.1-8B-Instruct
model:       meta-llama/Llama-3.1-8B-Instruct
class:       LlamaForCausalLM (model_type=llama)
layers:      32
hidden size: 4096
heads:       32
vocab:       128256
note:        could not build the model on the meta device (ModuleNotFoundError: Could not import module 'LlamaForCausalLM'. Are this object's requirements defined correctly?); the numbers come from config.json and tensor-type support is unverified

tensor_type      hf
residual_stream  unverified
attn_out         unverified
mlp_out          unverified
kv               no [1]
final_norm       no [2]
  [1] the KV cache isn't a per-layer forward-pass tensor with one row per token; the HF adapter doesn't capture it
  [2] the HF adapter doesn't hook the model-level final norm (the vLLM adapter does)

Probe it with layers 0..31, e.g. layers: [16], position: "prompt[-1]"



In [8]:
# Cell 6 — Save model inspection report locally and trigger download
import os
import subprocess
from google.colab import files

# We will write to the local temporary space which is always writable
output_filepath = "/content/model_inspection_report.txt"

MODEL_IDS = [
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "google/gemma-2-9b-it",
]

report_lines = []

for model_id in MODEL_IDS:
    header = f"\n" + "=" * 90 + f"\nMODEL: {model_id}\n"
    print(header)
    report_lines.append(header)

    result = subprocess.run(
        ["undercurrent", "inspect-model", model_id, "--backend", "hf"],
        text=True, capture_output=True
    )

    print(result.stdout)
    report_lines.append(result.stdout)

    if result.returncode != 0:
        err_msg = f"ERROR: {result.stderr}\n"
        print(err_msg)
        report_lines.append(err_msg)

# Save locally to Colab's workspace
with open(output_filepath, "w", encoding="utf-8") as f:
    f.writelines(report_lines)

print(f"\nSaved locally to: {output_filepath}")
print("Downloading the file to your computer now...")

# Trigger browser download of the file
files.download(output_filepath)


MODEL: meta-llama/Llama-3.1-8B-Instruct

model:       meta-llama/Llama-3.1-8B-Instruct
class:       LlamaForCausalLM (model_type=llama)
layers:      32
hidden size: 4096
heads:       32
vocab:       128256
note:        could not build the model on the meta device (ModuleNotFoundError: Could not import module 'LlamaForCausalLM'. Are this object's requirements defined correctly?); the numbers come from config.json and tensor-type support is unverified

tensor_type      hf
residual_stream  unverified
attn_out         unverified
mlp_out          unverified
kv               no [1]
final_norm       no [2]
  [1] the KV cache isn't a per-layer forward-pass tensor with one row per token; the HF adapter doesn't capture it
  [2] the HF adapter doesn't hook the model-level final norm (the vLLM adapter does)

Probe it with layers 0..31, e.g. layers: [16], position: "prompt[-1]"


MODEL: Qwen/Qwen2.5-1.5B-Instruct

model:       Qwen/Qwen2.5-1.5B-Instruct
class:       Qwen2ForCausalLM (model_type=qw

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# Cell 6 — smoke-test Undercurrent on a tiny public model first
# This validates the package/API path before spending time on Llama.
from undercurrent.core import ActivationRecord, probe
from undercurrent.model import ProbedModel

UC_SPEC = """
version: "1"
extraction_points:
  - name: last_prompt_token
    layers: 2
    tensor_type: residual_stream
    position: "prompt[-1]"
    probe_type: norm_probe
    probe_kind: single_shot
    execution_mode: inline
"""

@probe("norm_probe", threshold=1e9)
def norm_probe(record: ActivationRecord) -> float:
    return float(record.tensor.float().norm().item())

with ProbedModel.from_pretrained(
    "openai-community/gpt2",
    spec=UC_SPEC,
    probes={"norm_probe": norm_probe},
) as uc_model:
    uc_out = uc_model.generate("A short test prompt.", max_new_tokens=1, temperature=0)

print("Undercurrent smoke test output:", repr(uc_out.text))
print("Probe verdict:", uc_out.probe_results["last_prompt_token"].verdict)
assert uc_out.aborted is False

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Undercurrent smoke test output: '\n'
Probe verdict: {'flagged': False, 'max_score': 58.85881042480469, 'n': 1}
